## Essai Forecasting numéro 100001

In [1]:
import pandas as pd
import pickle
from typing import NamedTuple, Dict, List, Any
from tqdm import tqdm  # For progress bar during calculations

# Biogeme 
import biogeme.database as db
from biogeme.biogeme import BIOGEME
from biogeme.expressions import Expression, Variable, Derive
from biogeme.models import loglogit, logit, nested, lognested

# models pref
from models.logit_lmpc12_model3 import V_3, chosen_alternative
from models.logit_lmpc12_model4 import results_nested, nests

In [2]:
# Charger les données
df = pd.read_csv('models/lpmc12.dat', sep='\t')

# Scénario (i): Ajout de 1,5 £ au coût pour les utilisateurs de voiture
df_scenario1 = df.copy()
df_scenario1['cost_driving_fuel'] += 1.5

# Scénario (ii): Réduction de 20 % du coût des transports publics
df_scenario2 = df.copy()
df_scenario2['cost_transit'] *= 0.8

# Définir les tailles de population pour chaque segment
census = {
    'female_44_less': 2841376,
    'female_45_more': 1519948,
    'male_44_less': 2926408,
    'male_45_more': 1379198,
}

population_size = sum(census.values())

# Définir les filtres pour chaque segment
filters = {
    'female_44_less': (df['female'] == 1) & (df['age'] <= 44),
    'female_45_more': (df['female'] == 1) & (df['age'] > 44),
    'male_44_less': (df['female'] == 0) & (df['age'] <= 44),
    'male_45_more': (df['female'] == 0) & (df['age'] > 44),
}

# Count the sample size in each stratum
sample_segments = {
    segment_name: segment_rows.sum() for segment_name, segment_rows in filters.items()
}
print(f'Sample segments: {sample_segments}')

# Total sample size
total_sample = sum(sample_segments.values())
print(f'Sample size: {total_sample}')

weights = {
    segment_name: census[segment_name] * total_sample / (segment_size * population_size)
    for segment_name, segment_size in sample_segments.items()
}

# Ajouter les poids dans les datasets
for segment_name, segment_rows in filters.items():
    df.loc[segment_rows, 'weight'] = weights[segment_name]
    df_scenario1.loc[segment_rows, 'weight'] = weights[segment_name]
    df_scenario2.loc[segment_rows, 'weight'] = weights[segment_name]

# Définir les modèles et scénarios
model_market_share: dict[str, List] = {
    "Model 4": [V_3, results_nested],
    "Model 4 scenario 1": [V_3, results_nested],
    "Model 4 scenario 2": [V_3, results_nested],
}

# Créer des bases de données pour chaque scénario
database_original = db.Database("Original", df)
database_scenario1 = db.Database("Scenario 1", df_scenario1)
database_scenario2 = db.Database("Scenario 2", df_scenario2)

# Associer les bases de données à chaque modèle
databases = {
    "Model 4": database_original,
    "Model 4 scenario 1": database_scenario1,
    "Model 4 scenario 2": database_scenario2,
}

# Classe pour les résultats des parts de marché
class IndicatorTuple(NamedTuple):
    value: float
    lower: float
    upper: float

# Fonction pour calculer les parts de marché pondérées
def market_share(utilities: dict[int, Expression], results, database) -> dict[str, IndicatorTuple]:
    
    simulate = {
        'weight': Variable('weight'),
        'Prob. walk': nested(utilities, None, nests, 1),
        'Prob. cycle': nested(utilities, None, nests, 2),
        'Prob. PT': nested(utilities, None, nests, 3),
        'Prob. car': nested(utilities, None, nests, 4),
    }

    biosim = BIOGEME(database, simulate)
    simulated_values = biosim.simulate(results.get_beta_values())

    logprob = lognested(utilities, None, nests, chosen_alternative)
    biogeme = BIOGEME(database, logprob)
    betas = biogeme.free_beta_names
    sensitivity_betas = results.get_betas_for_sensitivity_analysis(
        betas, use_bootstrap=False
    )
    left, right = biosim.confidence_intervals(sensitivity_betas, 0.9)

    market_shares = {}
    
    for alt_name, prob_name in [
        ("Walking", "Prob. walk"),
        ("Cycling", "Prob. cycle"),
        ("Public transportation", "Prob. PT"),
        ("Car", "Prob. car"),
    ]:
        weighted_prob = simulated_values["weight"] * simulated_values[prob_name]
        left_weighted = left["weight"] * left[prob_name]
        right_weighted = right["weight"] * right[prob_name]
        
        market_share_value = weighted_prob.sum() / simulated_values["weight"].sum()
        market_share_lower = left_weighted.sum() / left["weight"].sum()
        market_share_upper = right_weighted.sum() / right["weight"].sum()
        market_shares[alt_name] = IndicatorTuple(value=market_share_value, lower=market_share_lower, upper=market_share_upper)

    return market_shares

# Fichier pickle pour sauvegarder ou charger les résultats
file_name = 'market_shares_forecastingV2.pickle'

try:
    # Lire les parts de marché depuis le fichier
    with open(file_name, 'rb') as f:
        all_market_shares = pickle.load(f)
        print(f'Market shares read from {file_name}')
except FileNotFoundError:
    # Calculer les parts de marché si le fichier n'existe pas
    all_market_shares: dict[str, Any] = {}
    for model, (utilities, results) in tqdm(model_market_share.items()):
        database = databases[model]
        all_market_shares[model] = market_share(utilities, results, database)
    
    # Sauvegarder les parts de marché
    with open(file_name, 'wb') as f:
        pickle.dump(all_market_shares, f)
    print(f'Market shares calculated and saved in {file_name}')


Sample segments: {'female_44_less': np.int64(1631), 'female_45_more': np.int64(1034), 'male_44_less': np.int64(1451), 'male_45_more': np.int64(884)}
Sample size: 5000


100%|██████████| 3/3 [04:09<00:00, 83.23s/it]

Market shares calculated and saved in market_shares_forecastingV2.pickle


In [3]:
print(all_market_shares["Model 4"])
print(all_market_shares["Model 4 scenario 1"])
print(all_market_shares["Model 4 scenario 2"])

{'Walking': IndicatorTuple(value=np.float64(0.16926692227752185), lower=np.float64(0.14623525910257076), upper=np.float64(0.18802420905120312)), 'Cycling': IndicatorTuple(value=np.float64(0.02919112349750728), lower=np.float64(0.020019554536691696), upper=np.float64(0.03491852373763166)), 'Public transportation': IndicatorTuple(value=np.float64(0.361891207822671), lower=np.float64(0.318474477191563), upper=np.float64(0.4974573760896563)), 'Car': IndicatorTuple(value=np.float64(0.43965074640229995), lower=np.float64(0.3138861920732546), upper=np.float64(0.48900056455419744))}
{'Walking': IndicatorTuple(value=np.float64(0.17529698384101083), lower=np.float64(0.1510596233257534), upper=np.float64(0.19543345412157778)), 'Cycling': IndicatorTuple(value=np.float64(0.03064460690037343), lower=np.float64(0.019351253935982856), upper=np.float64(0.03924310226195994)), 'Public transportation': IndicatorTuple(value=np.float64(0.4063040736288914), lower=np.float64(0.33092221700927144), upper=np.flo

In [4]:
# Afficher les résultats
for model, shares in all_market_shares.items():
    print(f"\nMarket shares for {model}:")
    for mode, share in shares.items():
        print(f"  {mode}: {share.value:.2%}")


Market shares for Model 4:
  Walking: 16.93%
  Cycling: 2.92%
  Public transportation: 36.19%
  Car: 43.97%

Market shares for Model 4 scenario 1:
  Walking: 17.53%
  Cycling: 3.06%
  Public transportation: 40.63%
  Car: 38.78%

Market shares for Model 4 scenario 2:
  Walking: 16.89%
  Cycling: 2.90%
  Public transportation: 37.20%
  Car: 43.02%


## Public transportation total revenu

In [5]:


def revenues(utilities: dict[int, Expression], results, database) -> IndicatorTuple:
    """Calculate the revenues of all alternatives, given the
    specification of the utility functions and the expression of the cost.

    :param cost: expression to calculate the cost of the public transportation.
    :param utilities: specification of the utility functions. It is a
        dict where the keys are the IDs of the alternatives, and the
        values are the expressions of the utility functions.
    :return: tuple containing the value of the revenues, as well as
        the lower and upper bound of the 90% confidence interval.
    """
    prob_pt = nested(utilities, None,nests, 3) # dans le doute on met 3 ici
    cost = Variable('cost_transit')
    simulate = {
        'weight': Variable('weight'),
        'Revenues PT': prob_pt * cost,
    }
    biosim = BIOGEME(database, simulate)
    simulated_values = biosim.simulate(results.get_beta_values())

    # We also calculate confidence intervals for the calculated quantities
    logprob = lognested(utilities, None, nests, chosen_alternative)
    biogeme = BIOGEME(database, logprob)
    betas = biogeme.free_beta_names
    sensitivity_betas = results.get_betas_for_sensitivity_analysis(betas, use_bootstrap = False)
    left, right = biosim.confidence_intervals(sensitivity_betas, 0.9)

    # Revenues are calculated using the weighted mean of the individual quantities
    simulated_values['Weighted revenues PT'] = (
        simulated_values['weight'] * simulated_values['Revenues PT']
    )
    left['Weighted revenues PT'] = left['weight'] * left['Revenues PT']
    right['Weighted revenues PT'] = right['weight'] * right['Revenues PT']

    revenues_pt = simulated_values['Weighted revenues PT'].mean()
    revenues_pt_left = left['Weighted revenues PT'].mean()
    revenues_pt_right = right['Weighted revenues PT'].mean()

    return IndicatorTuple(
        value=revenues_pt, lower=revenues_pt_left, upper=revenues_pt_right
    )



In [6]:
# now, how to use it ?
# different costs for each scenario
costs = {
    "Model 4": [V_3, results_nested, database_original],
    "Model 4 scenario 1": [V_3, results_nested, database_scenario1],
    "Model 4 scenario 2": [V_3, results_nested, database_scenario2],
}
file_name = 'revenues_pt.pickle'
try:
    with open(file_name, 'rb') as f:
        all_rev = pickle.load(f)
        print(f'Results read from {file_name}')
except FileNotFoundError:
    all_rev: dict[str, Any] = {}
    for model, (utilities, results, database) in tqdm(costs.items()):
        database = databases[model]
        all_rev[model] = revenues(utilities, results, database)

    with open(file_name, 'wb') as f:
        pickle.dump(all_rev, f)
    print(f'Results calculated and saved in {file_name}')

100%|██████████| 3/3 [01:24<00:00, 28.26s/it]

Results calculated and saved in revenues_pt.pickle


In [7]:
# Afficher les résultats
for model, (revenu, lower, upper) in all_rev.items():
    print(f"\nRevenu for {model}:")
    print(f"  Public Transportation: {revenu:.2} [{lower:.2}, {upper:.2}]")


Revenu for Model 4:
  Public Transportation: 0.72 [0.61, 0.89]

Revenu for Model 4 scenario 1:
  Public Transportation: 0.79 [0.68, 0.94]

Revenu for Model 4 scenario 2:
  Public Transportation: 0.6 [0.49, 0.75]


### Average value of time

In [8]:
# beta_tt
# for car and pt
# pt : boxcox_time_3
# car : boxcox_time_4
# get the results 
# Charger les données

vot_pt = Derive(V_3[3], 'segmented_b_time_3') / Derive(V_3[3], 'cost_transit')
vot_car = Derive(V_3[4], 'segmented_b_time_4') / Derive(V_3[4], 'cost_car')

def value_time(utilities: dict[int, Expression], database):
    
    simulate = {
        'weight': Variable('weight'),
        'WTP PT time': vot_pt,
        'WTP CAR time': vot_car,
    }

    biosim = BIOGEME(database, simulate)
    logprob = lognested(utilities, None, nests, chosen_alternative)
    biogeme = BIOGEME(database, logprob)
    results = biogeme.estimate(recycle=False)
    simulated_values = biosim.simulate(results.get_beta_values())
    
    return results, simulated_values

In [9]:
# scenario 1
results1, simulated_values1 = value_time(V_3, database_scenario1)
display(results1.get_estimated_parameters())
avg_vot_pt = (
    simulated_values1['WTP PT time'] * simulated_values1['weight']
).sum() / simulated_values1['weight'].sum()
print(f'Average value of time for PT: {60 * avg_vot_pt:.3g} £/hour')

avg_vot_car = (
    simulated_values1['WTP CAR time'] * simulated_values1['weight']
).sum() / simulated_values1['weight'].sum()
print(f'Average value of time for car: {60 * avg_vot_car:.3g} £/hour')

You have not defined a name for the model. The output .py are named from the model name. The default is [biogemeModelDefaultName]
Parameter beta_travel_time not present in the model.
Parameter beta_travel_time_2 not present in the model.
Parameter beta_travel_time_2_non-work-education-related not present in the model.
Parameter beta_travel_time_non-work-education-related not present in the model.
Parameter constant_2 not present in the model.
Parameter mu_motorized not present in the model.


KeyError: 'segmented_b_time_3'

In [ ]:
# scenario 2
results2 = value_time(V_3, database_scenario2)
display(results2.get_estimated_parameters())